# Semáforo de salud financiera empresarial

**Proyecto integrador** — Inteligencia de Negocios · Análisis de Datos · Aprendizaje Computacional
Instituto Tecnológico Metropolitano (ITM) · Equipo: Julián, Santiago, Jacobo y Oscar

## 0. Problema de negocio

Cuando una empresa entra en dificultades financieras, el problema suele verse en sus estados
financieros antes de que se vuelva una crisis: la liquidez cae, el endeudamiento sube o el
patrimonio se vuelve negativo. Detectar esas señales a tiempo le sirve a varios actores:

- **Entes de supervisión** (como la Superintendencia de Sociedades), para priorizar a qué
  empresas hacerles seguimiento.
- **Bancos y proveedores**, para decidir a quién otorgar crédito o plazos de pago.
- **Las mismas empresas**, para compararse con su sector y corregir el rumbo.

**Pregunta de negocio:** con el balance de una empresa en el año *t*, ¿podemos anticipar si
estará en **deterioro financiero** en el año *t+1*?

**Fuentes de datos** (públicas, Portal de Datos Abiertos de Colombia):

1. *Estados Financieros NIIF – Estado de Situación Financiera* (Supersociedades): balances de las
   empresas, una fila por empresa, cuenta y fecha de corte.
2. *10.000 Empresas más Grandes del País* (Supersociedades): sector, región, ciudad, ingresos y
   ganancia de cada empresa.

**Estructura del notebook:**

| Sección | Etapa |
|---|---|
| 1 | ETL: extracción, limpieza, integración y modelo estrella |
| 2 | Análisis exploratorio e inferencia estadística |
| 3 | Preparación de datos y modelos de aprendizaje supervisado |
| 4 | Conclusiones, recomendaciones y consideraciones éticas |

El dashboard de Power BI está en la carpeta `dashboard/`.

## 1. ETL — Extracción, transformación y carga

### 1.1 Extracción de datos por API

Las dos bases se publican en el Portal de Datos Abiertos, que usa la plataforma **Socrata**.
Socrata permite consultarlas con un lenguaje parecido a SQL (SoQL) directamente en la URL:

- `$select`: columnas a traer.
- `$where`: filtros.
- `$limit` y `$offset`: paginación.

Descargar por API, en lugar de bajar un archivo a mano, deja el proceso **automatizado y
reproducible**: cualquiera del equipo puede volver a ejecutar esta sección y obtener los mismos
datos.

La base de balances tiene cerca de **19,6 millones de filas**, así que no se descarga completa.
Se filtra en el servidor para traer solo lo que el proyecto necesita.

#### Configuración

Las rutas se construyen de forma **relativa** a la raíz del repositorio. Así el notebook
funciona en el computador de cualquier integrante.

In [1]:
import os
import io
import time
from pathlib import Path

import pandas as pd
import requests

# Raíz del repositorio: si el notebook se abre desde /notebooks, se sube un nivel
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = RAIZ / "data" / "raw"
DATA_PROCESSED = RAIZ / "data" / "processed"
DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# Identificadores de las bases en datos.gov.co
URL_BASE = "https://www.datos.gov.co/resource/{}.csv"
ID_BALANCES = "pfdp-zks5"   # Estados Financieros NIIF - Estado de Situación Financiera
ID_EMPRESAS = "6cat-2gcs"   # 10.000 Empresas más Grandes del País

# Años de corte a descargar (balances de cierre de año)
ANIOS = list(range(2019, 2026))

# Token opcional de datos.gov.co (variable de entorno, nunca escrito en el código)
APP_TOKEN = os.getenv("DATOS_GOV_TOKEN")

print("Raíz del proyecto:", RAIZ.name)
print("Token configurado:", "sí" if APP_TOKEN else "no (la descarga funciona, pero puede ser más lenta)")

Raíz del proyecto: Final-AnalisisDatos
Token configurado: no (la descarga funciona, pero puede ser más lenta)


#### Función de descarga paginada

La API devuelve como máximo un número fijo de filas por consulta. La función pide los datos por
páginas de 50.000 filas hasta que una página llega vacía.

- El parámetro `$order=:id` ordena por el identificador interno de cada fila. Así ninguna fila
  se repite ni se pierde entre páginas.
- Si una página falla por un error de red, se reintenta hasta 3 veces antes de detenerse.
- Todo se lee como texto (`dtype=str`). La conversión de tipos se hace en la etapa de limpieza.

In [2]:
def descargar_socrata(id_base, params, tam_pagina=50_000, reintentos=3):
    """Descarga una consulta SoQL completa, página por página, y la devuelve como DataFrame."""
    url = URL_BASE.format(id_base)
    encabezados = {"X-App-Token": APP_TOKEN} if APP_TOKEN else {}
    paginas, offset = [], 0

    while True:
        # Orden por defecto: identificador interno de la fila (se puede cambiar con params)
        consulta = {"$order": ":id", **params, "$limit": tam_pagina, "$offset": offset}
        for intento in range(1, reintentos + 1):
            try:
                resp = requests.get(url, params=consulta, headers=encabezados, timeout=300)
                resp.raise_for_status()
                break
            except requests.RequestException as error:
                if intento == reintentos:
                    raise
                print(f"  Fallo de conexión en la página (offset={offset}), "
                      f"reintento {intento} de {reintentos - 1}...")
                time.sleep(5 * intento)

        if not resp.text.strip():
            break
        pagina = pd.read_csv(io.StringIO(resp.text), dtype=str)
        if pagina.empty:
            break
        paginas.append(pagina)
        offset += tam_pagina
        print(f"  {offset - tam_pagina + len(pagina):,} filas descargadas")
        if len(pagina) < tam_pagina:
            break

    return pd.concat(paginas, ignore_index=True) if paginas else pd.DataFrame()

#### Base 2: 10.000 Empresas más Grandes del País

Esta base se descarga primero y completa, porque es pequeña (50.000 filas: 10.000 empresas por
cada año de corte entre 2021 y 2025). Define el **universo de análisis**: solo se trabajará con
las empresas que aparecen en ella, que son las que tienen información de sector y ubicación.

Una revisión previa de la API mostró que el NIT viene como texto con decimales (por ejemplo
`"899999068.00"`). Aquí solo se le quita ese `.00` para poder filtrar la otra base. La
normalización completa del NIT se hace en la etapa de integración.

In [3]:
empresas_raw = descargar_socrata(ID_EMPRESAS, params={})
empresas_raw.to_csv(DATA_RAW / "empresas_10000_raw.csv", index=False)

print("Dimensiones:", empresas_raw.shape)
print("Columnas:", list(empresas_raw.columns))
empresas_raw.head()

  50,000 filas descargadas
Dimensiones: (50000, 14)
Columnas: ['nit', 'raz_n_social', 'supervisor', 'regi_n', 'departamento_domicilio', 'ciudad_domicilio', 'ciiu', 'macrosector', 'ingresos_operacionales', 'ganancia_p_rdida', 'total_activos', 'total_pasivos', 'total_patrimonio', 'a_o_de_corte']


,nit,raz_n_social,supervisor,regi_n,departamento_domicilio,ciudad_domicilio,ciiu,macrosector,ingresos_operacionales,ganancia_p_rdida,total_activos,total_pasivos,total_patrimonio,a_o_de_corte
0,899999068.00,ECOPETROL S.A.,SUPERFINANCIERA,BOGOTÁ - CUNDINAMARCA,BOGOTA D.C.,"BOGOTA, D.C.",610,MINERO,$ 100.59,$ 9.03,$ 198.11,$ 114.35,$ 83.76,2025.00
1,830095213.00,ORGANIZACIÓN TERPEL S.A.,SUPERFINANCIERA,BOGOTÁ - CUNDINAMARCA,BOGOTA D.C.,"BOGOTA, D.C.",4661,COMERCIO,$ 26.40,$ 0.63,$ 8.11,$ 4.70,$ 3.41,2025.00
2,900112515.00,REFINERIA DE CARTAGENA S.A.,SUPERSERVICIOS,REGIÓN CARIBE,BOLIVAR,CARTAGENA,1921,MANUFACTURA,$ 22.40,-$ 0.82,$ 34.95,$ 11.57,$ 23.38,2025.00
3,900276962.00,D1 S.A.S.,SUPERSOCIEDADES,BOGOTÁ - CUNDINAMARCA,BOGOTA D.C.,"BOGOTA, D.C.",4711,COMERCIO,$ 21.61,$ 0.42,$ 6.94,$ 6.72,$ 0.21,2025.00
4,890904996.00,EMPRESAS PÚBLICAS DE MEDELLÍN E.S.P.,SUPERFINANCIERA,ANTIOQUIA,ANTIOQUIA,MEDELLÍN,3513,SERVICIOS,$ 20.29,$ 4.88,$ 71.08,$ 36.11,$ 34.97,2025.00


In [4]:
# NIT sin decimales, solo para filtrar los balances
empresas_raw["nit_filtro"] = empresas_raw["nit"].str.strip().str.replace(r"\.0+$", "", regex=True)
NITS_UNIVERSO = set(empresas_raw["nit_filtro"].dropna())

print("Filas por año de corte:")
print(empresas_raw["a_o_de_corte"].value_counts().sort_index())
print(f"\nEmpresas distintas en el universo: {len(NITS_UNIVERSO):,}")

Filas por año de corte:
a_o_de_corte
2021.00    10000
2022.00    10000
2023.00    10000
2024.00    10000
2025.00    10000
Name: count, dtype: int64

Empresas distintas en el universo: 15,032


> **Interpretación:** la base tiene 50.000 registros: 10.000 empresas por año de corte entre 2021 y 2025. Como muchas empresas aparecen en varios años, el universo queda en **15.032 empresas distintas**. Todas las columnas llegan como texto: el NIT y el año de corte traen decimales (`899999068.00`, `2025.00`) y los montos traen el signo `$` y están expresados en billones de pesos, así que se convierten en la etapa de limpieza. El 87,5 % de los registros corresponde a empresas vigiladas por Supersociedades; el resto lo vigilan otras superintendencias (Supersalud, Supertransporte, Supervigilancia, Superservicios y Superfinanciera).

#### Inventario de cuentas disponibles

En la base de balances, cada cuenta contable aparece en la columna `concepto` con su nombre
textual. Antes de descargar hay que saber **cómo se llaman exactamente** las cuentas que
necesitamos.

Para no consultar los 19,6 millones de filas, el inventario se hace solo sobre el corte del
31 de diciembre de 2024. El resultado se guarda en `data/raw/conceptos_disponibles.csv` como
evidencia del proceso.

Filtros que se aplican, tanto aquí como en la descarga:

- `periodo = 'Periodo Actual'`: cada reporte también trae el año anterior como comparativo.
  Ese dato ya está en el reporte del año anterior, así que se descarta para no duplicarlo.
- `punto_entrada` sin las palabras *Consolidado* ni *Combinado*: se usan los estados
  **individuales** y **separados** de cada empresa. Los consolidados y combinados suman varias
  entidades (por ejemplo, una empresa y sus subordinadas), así que no son comparables con los
  demás.

In [5]:
FILTRO_COMUN = ("periodo = 'Periodo Actual' AND punto_entrada not like '%Consolidado%' "
                "AND punto_entrada not like '%Combinado%'")

conceptos = descargar_socrata(
    ID_BALANCES,
    params={
        "$select": "concepto, count(*) AS n",
        "$where": f"fecha_corte = '2024-12-31T00:00:00.000' AND {FILTRO_COMUN}",
        "$group": "concepto",
        "$order": "n DESC",
    },
)
conceptos["n"] = conceptos["n"].astype(int)
conceptos = conceptos.sort_values("n", ascending=False).reset_index(drop=True)
conceptos.to_csv(DATA_RAW / "conceptos_disponibles.csv", index=False)

print(f"Cuentas distintas en el corte 2024: {len(conceptos):,}")
conceptos.head(60)

  71 filas descargadas
Cuentas distintas en el corte 2024: 71


,concepto,n
0,Total de activos,29427
1,Patrimonio total,29427
2,Efectivo y equivalentes al efectivo,29427
3,Total pasivos,29427
4,Total de patrimonio y pasivos,29427
5,Total activos corrientes distintos de los acti...,29427
6,Activos corrientes totales,29427
7,Capital emitido,29426
8,Ganancias acumuladas,29408
9,Pasivos corrientes totales,29221


#### Selección de cuentas

Para calcular los indicadores financieros del proyecto se necesitan las 10 cuentas de la tabla
siguiente. Una misma cuenta puede aparecer con nombres distintos según la taxonomía (NIIF Plenas
o NIIF Pymes), por eso cada cuenta tiene una **lista** de nombres posibles.

| Cuenta | Para qué se usa |
|---|---|
| Activo corriente | Liquidez, prueba ácida, capital de trabajo |
| Activo total | Endeudamiento, solvencia |
| Pasivo corriente | Liquidez, prueba ácida, capital de trabajo |
| Pasivo total | Endeudamiento, validación contable |
| Patrimonio total | Solvencia, variable objetivo, validación contable |
| Efectivo y equivalentes | Liquidez inmediata |
| Inventarios | Prueba ácida |
| Deudores comerciales | Calidad del activo corriente |
| Capital emitido | Contexto patrimonial |
| Ganancias acumuladas | Historial de resultados |

La celda que sigue a la de las listas verifica qué nombres existen realmente en el inventario.
**Si alguna cuenta sale sin coincidencias**, se busca su nombre en
`conceptos_disponibles.csv` y se agrega a la lista correspondiente antes de descargar.

In [6]:
# Nombres según la taxonomía NIIF en español. "Patrimonio total" y "Capital emitido" se
# verificaron en la API; los demás se confirman con la celda siguiente.
CUENTAS = {
    "activo_corriente":     ["Activos corrientes totales"],
    "activo_total":         ["Total de activos"],
    "pasivo_corriente":     ["Pasivos corrientes totales"],
    "pasivo_total":         ["Total pasivos"],
    "patrimonio_total":     ["Patrimonio total"],
    "efectivo":             ["Efectivo y equivalentes al efectivo"],
    "inventarios":          ["Inventarios corrientes", "Inventarios"],
    "deudores_comerciales": ["Cuentas comerciales por cobrar y otras cuentas por cobrar corrientes"],
    "capital_emitido":      ["Capital emitido"],
    "ganancias_acumuladas": ["Ganancias acumuladas"],
}

In [7]:
disponibles = set(conceptos["concepto"])
faltantes = []
for cuenta, nombres in CUENTAS.items():
    encontrados = [n for n in nombres if n in disponibles]
    estado = "OK" if encontrados else "SIN COINCIDENCIA"
    print(f"{cuenta:22s} {estado:17s} {encontrados}")
    if not encontrados:
        faltantes.append(cuenta)

# Ayuda para corregir: nombres del inventario que contienen una palabra clave de cada faltante
PALABRAS_CLAVE = {
    "activo_corriente": "activos corrientes", "activo_total": "activos",
    "pasivo_corriente": "pasivos corrientes", "pasivo_total": "pasivos",
    "patrimonio_total": "patrimonio", "efectivo": "efectivo", "inventarios": "inventario",
    "deudores_comerciales": "por cobrar", "capital_emitido": "capital",
    "ganancias_acumuladas": "ganancias acumuladas",
}
for cuenta in faltantes:
    clave = PALABRAS_CLAVE[cuenta]
    sugeridos = conceptos[conceptos["concepto"].str.lower().str.contains(clave, na=False)]
    print(f"\nPosibles nombres para '{cuenta}':")
    print(sugeridos.head(10).to_string(index=False))

activo_corriente       OK                ['Activos corrientes totales']
activo_total           OK                ['Total de activos']
pasivo_corriente       OK                ['Pasivos corrientes totales']
pasivo_total           OK                ['Total pasivos']
patrimonio_total       OK                ['Patrimonio total']
efectivo               OK                ['Efectivo y equivalentes al efectivo']
inventarios            OK                ['Inventarios corrientes']
deudores_comerciales   OK                ['Cuentas comerciales por cobrar y otras cuentas por cobrar corrientes']
capital_emitido        OK                ['Capital emitido']
ganancias_acumuladas   OK                ['Ganancias acumuladas']


> **Interpretación:** las 10 cuentas necesarias aparecen con el mismo nombre en el corte de 2024, así que no hubo que ajustar el diccionario `CUENTAS`. En ese corte hay 71 cuentas distintas y las principales (activos, pasivos, patrimonio, efectivo, capital) las reportan unas 29.400 empresas. La excepción es inventarios corrientes, que aparece en 18.795 reportes (64 %), porque muchas empresas de servicios no manejan inventario.

#### Base 1: balances por año

La descarga se hace **un año a la vez**. Así cada consulta es más liviana y se evitan los tiempos
de espera agotados en el servidor. Después de cada año se conservan solo las empresas del
universo definido por la base 2, lo que reduce mucho el tamaño del archivo final.

Por cada fila se descargan:

- `nit`, `fecha_corte` y `concepto`.
- `valor`: sigue como texto; se convierte en la limpieza.
- `punto_entrada` y `taxonomia`: indican el tipo de estado financiero.
- `numero_radicado` y `codigo_instancia`: sirven en la limpieza para detectar
  retransmisiones del mismo reporte.

In [8]:
nombres_cuentas = sorted({n for lista in CUENTAS.values() for n in lista})
lista_soql = ", ".join("'" + n.replace("'", "''") + "'" for n in nombres_cuentas)
columnas = "nit, fecha_corte, punto_entrada, taxonomia, numero_radicado, codigo_instancia, concepto, valor"

balances_por_anio = []
for anio in ANIOS:
    print(f"Año {anio}:")
    df_anio = descargar_socrata(
        ID_BALANCES,
        params={
            "$select": columnas,
            "$where": (f"fecha_corte = '{anio}-12-31T00:00:00.000' AND {FILTRO_COMUN} "
                       f"AND concepto in ({lista_soql})"),
        },
    )
    if df_anio.empty:
        print("  sin datos")
        continue
    df_anio = df_anio[df_anio["nit"].str.strip().isin(NITS_UNIVERSO)]
    print(f"  {len(df_anio):,} filas de empresas del universo")
    balances_por_anio.append(df_anio)

balances_raw = pd.concat(balances_por_anio, ignore_index=True)
balances_raw.to_csv(DATA_RAW / "balances_raw.csv", index=False)
print("\nDimensiones finales:", balances_raw.shape)

Año 2019:
  50,000 filas descargadas
  100,000 filas descargadas
  150,000 filas descargadas
  200,000 filas descargadas
  246,654 filas descargadas
  93,378 filas de empresas del universo
Año 2020:
  Fallo de conexión en la página (offset=0), reintento 1 de 2...
  50,000 filas descargadas
  100,000 filas descargadas
  150,000 filas descargadas
  200,000 filas descargadas
  250,000 filas descargadas
  279,152 filas descargadas
  98,023 filas de empresas del universo
Año 2021:
  50,000 filas descargadas
  100,000 filas descargadas
  150,000 filas descargadas
  200,000 filas descargadas
  250,000 filas descargadas
  280,975 filas descargadas
  103,061 filas de empresas del universo
Año 2022:
  50,000 filas descargadas
  100,000 filas descargadas
  150,000 filas descargadas
  200,000 filas descargadas
  250,000 filas descargadas
  300,000 filas descargadas
  350,000 filas descargadas
  400,000 filas descargadas
  450,000 filas descargadas
  500,000 filas descargadas
  550,000 filas descar

#### Resumen de la extracción

Se verifica qué se descargó: filas por año, empresas distintas por año y filas por cuenta. Si
algún año o alguna cuenta trae muy pocos datos, hay que revisar los filtros antes de pasar a la
limpieza.

In [9]:
balances_raw["anio"] = balances_raw["fecha_corte"].str[:4]

resumen = balances_raw.groupby("anio").agg(
    filas=("valor", "size"),
    empresas=("nit", "nunique"),
)
print(resumen, "\n")
print("Filas por cuenta:")
print(balances_raw["concepto"].value_counts())
print("\nTipos de estado financiero (punto de entrada):")
print(balances_raw["punto_entrada"].value_counts())
print("Empresas distintas con balance:", balances_raw["nit"].nunique())

       filas  empresas
anio                  
2019   93378      9495
2020   98023      9950
2021  103061     10435
2022  212064     10990
2023  114737     11647
2024  113098     11490
2025  115996     11822 

Filas por cuenta:
concepto
Total de activos                                                        86657
Total pasivos                                                           86657
Activos corrientes totales                                              86657
Efectivo y equivalentes al efectivo                                     86657
Patrimonio total                                                        86657
Capital emitido                                                         86653
Ganancias acumuladas                                                    86642
Pasivos corrientes totales                                              86586
Cuentas comerciales por cobrar y otras cuentas por cobrar corrientes    85845
Inventarios corrientes                                        

> **Interpretación:** Se descargaron balances de 12.761 de las 15.032 empresas del universo (84,9%). Las demás están vigiladas por otras superintendencias (Supersalud, Supertransporte, entre otras) y no reportan sus estados financieros a Supersociedades. El número de empresas con balance crece de 9.495 en 2019 a 11.822 en 2025. Las cuentas principales tienen alrededor de 86.600 registros; inventarios tiene menos (71.346) porque muchas empresas de servicios no manejan inventario. 2022 tiene casi el doble de filas que los demás años porque la fuente trae registros duplicados exactos; se eliminan en la etapa de limpieza.

**Archivos generados en `data/raw/`** (no se suben al repositorio; se regeneran ejecutando esta
sección):

| Archivo | Contenido |
|---|---|
| `empresas_10000_raw.csv` | Base 2 completa |
| `conceptos_disponibles.csv` | Inventario de cuentas del corte 2024 |
| `balances_raw.csv` | Balances 2019–2025 de las empresas del universo, formato largo |

*Siguiente paso:* limpieza y transformación de los balances.